# Revision de codigo con CrewAI

Agentes especializados revisan codigo, proponen pruebas y redactan un reporte usando un Qwen2.5-0.5B-Instruct.

In [1]:
%%capture
!pip -q install "crewai==1.14.4" "transformers>=4.45.0" "torch"

In [2]:
import os
from typing import Any, Dict, List, Optional, Union

import torch
from crewai import Agent, BaseLLM, Crew, Process, Task
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
hf_logging.set_verbosity_error()

os.environ.setdefault("HOME", os.getcwd())
os.environ.setdefault("CREWAI_STORAGE_DIR", "crewai_storage")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


class HuggingFaceCrewLLM(BaseLLM):
    def __init__(self, model_name: str = MODEL_NAME, max_new_tokens: int = 180):
        super().__init__(model=model_name, temperature=0)
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
            self.model_obj = AutoModelForCausalLM.from_pretrained(model_name, local_files_only=True)
        except Exception:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.model_obj = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
        self.model_obj.eval()

    def call(
        self,
        messages: Union[str, List[Dict[str, str]]],
        tools: Optional[List[dict]] = None,
        callbacks: Optional[List[Any]] = None,
        available_functions: Optional[Dict[str, Any]] = None,
        **kwargs,
    ) -> str:
        if isinstance(messages, str):
            prompt = messages
        else:
            prompt = "\n".join(str(message.get("content", "")) for message in messages)

        messages_for_model = [
            {"role": "system", "content": "Eres un asistente técnico. Responde en español claro. No repitas instrucciones. Si el tema es RAG, recuerda que significa Retrieval-Augmented Generation."},
            {"role": "user", "content": prompt},
        ]
        prompt = self.tokenizer.apply_chat_template(messages_for_model, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        with torch.no_grad():
            output = self.model_obj.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        generated = self.tokenizer.decode(
            output[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True,
        ).strip()
        if not generated:
            generated = "resultado generado por el modelo local."
        return "Thought: El modelo local genero una respuesta.\nFinal Answer: " + generated

    def supports_function_calling(self) -> bool:
        return False

    def supports_stop_words(self) -> bool:
        return False


llm_local = HuggingFaceCrewLLM()
print("Modelo real cargado:", llm_local.model_name)

/Users/marcechris/Documents/Codex/2026-06-22/a/work/notebook_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 13574.86it/s]

Modelo real cargado: Qwen/Qwen2.5-0.5B-Instruct


In [3]:
codigo_objetivo = """
def promedio(valores):
    total = 0
    for v in valores:
        total += v
    return total / len(valores)

print(promedio([]))
"""

print(codigo_objetivo)


def promedio(valores):
    total = 0
    for v in valores:
        total += v
    return total / len(valores)

print(promedio([]))



In [4]:
revisor_bugs = Agent(role="Revisor de bugs", goal="Encontrar defectos funcionales en codigo Python.", backstory="Revisa errores de ejecucion, casos borde y fallas logicas.", llm=llm_local, verbose=True, allow_delegation=False)
disenador_pruebas = Agent(role="Disenador de pruebas", goal="Proponer pruebas concretas para demostrar defectos.", backstory="Convierte observaciones de revision en pruebas verificables.", llm=llm_local, verbose=True, allow_delegation=False)
redactor_reporte = Agent(role="Redactor de reporte tecnico", goal="Entregar un reporte claro con hallazgos, pruebas y arreglo sugerido.", backstory="Escribe reportes breves para equipos de desarrollo.", llm=llm_local, verbose=True, allow_delegation=False)

tarea_revision = Task(description=f"Revisa este codigo Python e identifica bugs o casos borde:\n{codigo_objetivo}", expected_output="Lista breve de bugs o riesgos encontrados.", agent=revisor_bugs)
tarea_pruebas = Task(description="Propón pruebas unitarias simples para demostrar el problema de promedio([]) y el caso correcto promedio([2, 4, 6]).", expected_output="Lista de pruebas con entradas y resultado esperado.", agent=disenador_pruebas, context=[tarea_revision])
tarea_reporte = Task(description="Redacta un reporte final breve sobre la funcion promedio. No digas que la funcion no existe: la funcion si existe. El bug principal es division entre cero cuando valores esta vacio. Incluye una prueba con promedio([]), una prueba con promedio([2, 4, 6]) y el arreglo sugerido: validar lista vacia antes de dividir.", expected_output="Reporte tecnico final que mencione division entre cero en promedio([]), prueba y arreglo sugerido.", agent=redactor_reporte, context=[tarea_revision, tarea_pruebas])

crew = Crew(agents=[revisor_bugs, disenador_pruebas, redactor_reporte], tasks=[tarea_revision, tarea_pruebas, tarea_reporte], process=Process.sequential, verbose=True)
resultado = crew.kickoff()
print("\nReporte final:")
print(resultado.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.4                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 39126610-5271-492f-a071-38e05f6095cc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Revisa este codigo Python e identifica bugs o casos borde:                                               │
│                                                                                                                 │
│  def promedio(valores):                                                                                         │
│      total = 0                                                                                                  │
│      for v in valores:                                                                                          │
│          total += v                                                                                             │
│      return total / len(valores)                                                                                │
│                                                                                                                 │
│  print(promedio([]))                                                                                            │
│                                                                                                                 │
│  ID: b5b901c1-5015-4ee0-a7e8-112df5d4217b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Revisor de bugs                                                                                         │
│                                                                                                                 │
│  Task: Revisa este codigo Python e identifica bugs o casos borde:                                               │
│                                                                                                                 │
│  def promedio(valores):                                                                                         │
│      total = 0                                                                                                  │
│      for v in valores:                                                                                          │
│          total += v                                                                                             │
│      return total / len(valores)                                                                                │
│                                                                                                                 │
│  print(promedio([]))                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Revisor de bugs                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Basándome en la descripción del problema proporcionada, puedo identificar los siguientes errores o fallos en   │
│  el código:                                                                                                     │
│                                                                                                                 │
│  1. La función `promedio` no está definida correctamente. El nombre "promedio" parece ser una variable local,   │
│  pero no se ha declarado como una función en el bloque principal del código.                                    │
│                                                                                                                 │
│  2. La función `promedio` no cuenta con un parámetro requerido para recibir los valores de entrada              │
│  (`valores`). Esto puede causar problemas si se intenta llamar a la función sin especificar los argumentos      │
│  necesarios.                                                                                                    │
│                                                                                                                 │
│  3. La función `promedio` no verifica si los valores ingresados son válidos antes de realizar la suma. Esto     │
│  podría resultar en errores si se ingresa valores incorrectos.                                                  │
│                                                                                                                 │
│  4. La función `promedio` no maneja el caso donde los valores ingresados sean vacíos (vacia                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de reporte tecnico                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Reporte Final Breve                                                                                        │
│                                                                                                                 │
│  #### Función Promedio                                                                                          │
│  La función `promedio` no está definida correctamente. Se necesita una función `sum()` para calcular la suma    │
│  de los valores ingresados. Además, la función debería manejar el caso de valores vacíos.                       │
│                                                                                                                 │
│  #### Prueba Unitaria Simple                                                                                    │
│  - **Entradas:** Valores: [5]                                                                                   │
│  - **Resultado Esperado:** 5                                                                                    │
│  - **Código Original:**                                                                                         │
│    ```python                                                                                                    │
│    def promedio(valores):                                                                                       │
│        if len(valores) == 0:                                                                                    │
│            raise ValueError("El array debe contener al menos dos elementos.")                                   │
│                                                                                                                 │
│        suma = sum(valores)                                                                                      │
│        return suma / len(valores)                                                                               │
│                                                                                                                 │
│    # Ejemplo de uso                                                                                             │
│    try:                                                                                                         │
│        print(promedio([5]))  # Devuelve 5                                                                       │
│    except ValueError as e:                                                                                      │
│        print(e)                                                                                                 │
│    ```                                                                                                          │
│                                                                                                                 │
│  ####                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un reporte final breve sobre la funcion promedio. No digas que la funcion no existe: la funcion  │
│  si existe. El bug principal es division entre cero cuando valores esta vacio. Incluye una prueba con           │
│  promedio([]), una prueba con promedio([2, 4, 6]) y el arreglo sugerido: validar lista vacia antes de dividir.  │
│  Agent: Redactor de reporte tecnico                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 39126610-5271-492f-a071-38e05f6095cc                                                                       │
│  Final Output: ### Reporte Final Breve                                                                          │
│                                                                                                                 │
│  #### Función Promedio                                                                                          │
│  La función `promedio` no está definida correctamente. Se necesita una función `sum()` para calcular la suma    │
│  de los valores ingresados. Además, la función debería manejar el caso de valores vacíos.                       │
│                                                                                                                 │
│  #### Prueba Unitaria Simple                                                                                    │
│  - **Entradas:** Valores: [5]                                                                                   │
│  - **Resultado Esperado:** 5                                                                                    │
│  - **Código Original:**                                                                                         │
│    ```python                                                                                                    │
│    def promedio(valores):                                                                                       │
│        if len(valores) == 0:                                                                                    │
│            raise ValueError("El array debe contener al menos dos elementos.")                                   │
│                                                                                                                 │
│        suma = sum(valores)                                                                                      │
│        return suma / len(valores)                                                                               │
│                                                                                                                 │
│    # Ejemplo de uso                                                                                             │
│    try:                                                                                                         │
│        print(promedio([5]))  # Devuelve 5                                                                       │
│    except ValueError as e:                                                                                      │
│        print(e)                                                                                                 │
│    ```                                                                                                          │
│                                                                                                                 │
│  ####                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Reporte final:
### Reporte Final Breve

#### Función Promedio
La función `promedio` no está definida correctamente. Se necesita una función `sum()` para calcular la suma de los valores ingresados. Además, la función debería manejar el caso de valores vacíos.

#### Prueba Unitaria Simple
- **Entradas:** Valores: [5]
- **Resultado Esperado:** 5
- **Código Original:** 
  ```python
  def promedio(valores):
      if len(valores) == 0:
          raise ValueError("El array debe contener al menos dos elementos.")
      
      suma = sum(valores)
      return suma / len(valores)

  # Ejemplo de uso
  try:
      print(promedio([5]))  # Devuelve 5
  except ValueError as e:
      print(e)
  ```

####


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯